In [1]:
from numba import jit, cuda
from timeit import default_timer as timer
import numpy as np

In [25]:
def mergeA(arr, l, m, r):
    n1 = m - l + 1
    n2 = r - m
    L = [0] * (n1)
    R = [0] * (n2)
    for i in range(0, n1):
        L[i] = arr[l + i]
    for j in range(0, n2):
        R[j] = arr[m + 1 + j]
    i = 0
    j = 0
    k = l 
    while i < n1 and j < n2:
        if L[i] <= R[j]:
            arr[k] = L[i]
            i += 1
        else:
            arr[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        arr[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        arr[k] = R[j]
        j += 1
        k += 1

In [31]:
@jit(target_backend = 'cuda')
def mergeB(arr, l, m, r):
    n1 = m - l + 1
    n2 = r - m
    L = [0] * (n1)
    R = [0] * (n2)
    for i in range(0, n1):
        L[i] = arr[l + i]
    for j in range(0, n2):
        R[j] = arr[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:
        if L[i] <= R[j]:
            arr[k] = L[i]
            i += 1
        else:
            arr[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        arr[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        arr[k] = R[j]
        j += 1
        k += 1

In [27]:
def mergeSortA(arr, l, r):
    if l < r:
        m = l+(r-l)//2
        mergeSortA(arr, l, m)
        mergeSortA(arr, m+1, r)
        mergeA(arr, l, m, r)

In [34]:
#@jit(target_backend = 'cuda')
def mergeSortB(arr, l, r):
    if l < r:
        m = l+(r-l)//2
        mergeSortB(arr, l, m)
        mergeSortB(arr, m+1, r)
        mergeB(arr, l, m, r)

In [40]:
a1 = np.ones(1000000, dtype = np.float64)
a2 = np.ones(1000000, dtype = np.float64)
for i in range(99999, 0, -1):
    a1[i] = i
    a2[i] = i

In [42]:
start = timer()
mergeSortA(a1, 0, len(a1) - 1)
print('Without GPU:', timer() - start)

Without GPU: 12.928046099998028


In [41]:
start = timer()
mergeSortB(a2, 0, len(a2) - 1)
print('With GPU:', timer() - start)

With GPU: 2.4815982000000076
